<a href="https://colab.research.google.com/github/maty20arriagada/Caso-estudio-G11/blob/main/scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1) GPU y dependencias (versiones fijadas = smoke test local)
!nvidia-smi
%pip install -q "transformers==5.17.0" "trl==0.21.0" "peft==0.21.0" "datasets==5.0.1" "accelerate==1.15.0" bitsandbytes
%pip uninstall -y -q torchao

Sat Sep 26 22:20:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 2) Traer el proyecto (sube proyecto.zip, generado con: python datos/preparar_zip.py)
import os, sys, shutil, zipfile

if not os.path.isfile(os.path.join("proyecto", "entrenamiento", "qlora_entrenar.py")):
    if os.path.isdir("proyecto"):
        shutil.rmtree("proyecto")
    from google.colab import files
    print("Sube proyecto.zip (raiz del proyecto comprimida):")
    subido = files.upload()
    zip_path = next(iter(subido))
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            nombre = info.filename.replace("\\", "/")
            if not nombre or nombre.endswith("/") or "__pycache__" in nombre:
                continue
            ruta = os.path.join("proyecto", *nombre.split("/"))
            os.makedirs(os.path.dirname(ruta), exist_ok=True)
            with z.open(info) as src, open(ruta, "wb") as dst:
                shutil.copyfileobj(src, dst)
    hijos = [h for h in os.listdir("proyecto") if h != "__MACOSX"]
    if len(hijos) == 1 and os.path.isdir(os.path.join("proyecto", hijos[0])):
        raiz_zip = os.path.join("proyecto", hijos[0])
        for h in os.listdir(raiz_zip):
            shutil.move(os.path.join(raiz_zip, h), os.path.join("proyecto", h))
        os.rmdir(raiz_zip)

RAIZ = os.path.abspath("proyecto")
sys.path.insert(0, RAIZ)
os.chdir(RAIZ)
print("Proyecto en:", RAIZ)
!python entrenamiento/qlora_entrenar.py --help

Sube proyecto.zip (raiz del proyecto comprimida):


Saving proyecto.zip to proyecto.zip
Proyecto en: /content/proyecto
usage: qlora_entrenar.py [-h] [--modelo MODELO] [--datos DATOS]
                         [--epocas EPOCAS] [--lr LR] [--r R] [--alpha ALPHA]
                         [--dropout DROPOUT] [--batch BATCH] [--accum ACCUM]
                         [--max-length MAX_LENGTH] [--max-steps MAX_STEPS]
                         [--eval-limite EVAL_LIMITE] [--eval-lote EVAL_LOTE]
                         [--reanudar] [--sin-cuantizar] [--smoke]
                         [--salida SALIDA] [--semilla SEMILLA]

Fine-tuning QLoRA del extractor semantico

options:
  -h, --help            show this help message and exit
  --modelo MODELO
  --datos DATOS
  --epocas EPOCAS
  --lr LR
  --r R
  --alpha ALPHA
  --dropout DROPOUT
  --batch BATCH
  --accum ACCUM
  --max-length MAX_LENGTH
                        tokens por ejemplo; por defecto el maximo medido en
                        los datos + margen
  --max-steps MAX_STEPS
                  

In [4]:
# 3) Smoke test del codigo (modelo aleatorio: NO es evidencia)
!python entrenamiento/qlora_entrenar.py --smoke --max-steps 20 --epocas 1 --salida /tmp/smoke_adapter

datos: train=3200 dev=200 hard_dev=150 | campos=['titulo', 'hora_inicio', 'duracion_minutos', 'participantes', 'ubicacion', 'recordatorio_minutos']
config.json: 100% 727/727 [00:00<00:00, 4.02MB/s]
tokenizer_config.json: 100% 9.38k/9.38k [00:00<00:00, 26.9MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 71.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 125MB/s]

tokenizer.json: downloading bytes:   0% 0.00/11.4M [00:00<?, ?B/s]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:00<00:00, 5.82MB/s,  336kB/s  ]
tokenizer.json: reconstructing file: 100% 11.4M/11.4M [00:00<00:00, 19.5MB/s, 1.13MB/s  ]
trainable params: 32,768 || all params: 19,520,704 || trainable%: 0.1679
modo de entrenamiento: smoke (modelo diminuto aleatorio)
tokens por ejemplo: prompt<=524 | respuesta 42-70 (lo unico con loss) | total 519-591 | max_length=599
Truncating train dataset: 100% 3200/3200 [00:00<00:00, 79662.95 examples/s]
Truncating eval dataset: 100% 200/200 [00:00<00:00, 54902.86 examples/s]
masca

In [ ]:
# 4) Entrenamiento real -- checkpoints en Drive para sobrevivir desconexiones
from google.colab import drive
drive.mount('/content/drive')
SALIDA = "/content/drive/MyDrive/qwen3-4b-semantic-lora"
REANUDAR = False   # True si Colab se corto: sigue desde el ultimo checkpoint
!python entrenamiento/qlora_entrenar.py \
    --modelo Qwen/Qwen3-4B-Instruct-2507 \
    --epocas 3 \
    --salida {SALIDA} {"--reanudar" if REANUDAR else ""}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
datos: train=3200 dev=200 hard_dev=150 | campos=['titulo', 'hora_inicio', 'duracion_minutos', 'participantes', 'ubicacion', 'recordatorio_minutos']
Loading weights: 100% 398/398 [00:28<00:00, 13.84it/s]
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
modo de entrenamiento: 4-bit NF4 + LoRA (GPU)
tokens por ejemplo: prompt<=524 | respuesta 42-70 (lo unico con loss) | total 519-591 | max_length=599
Truncating train dataset: 100% 3200/3200 [00:00<00:00, 66864.14 examples/s]
Truncating eval dataset: 100% 200/200 [00:00<00:00, 65536.00 examples/s]
mascara de loss verificada: 59 de 561 tokens del ejemplo 0 entrenan (solo el JSON)
gradiente verificado: 252 tensores LoRA reciben gradiente (loss inicial 1.058)
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model confi

In [6]:
import torch; print(torch.__version__, torch.cuda.is_available())
!nvidia-smi -L

2.11.0+cu128 True
GPU 0: Tesla T4 (UUID: GPU-c863ee5d-7bcd-c828-081d-8e9f73a2ee47)


In [11]:
# parche: reactivar LoRA tras crear el SFTTrainer (trl 0.21 congela todo en 4-bit)
p = "/content/proyecto/entrenamiento/qlora_entrenar.py"
s = open(p, encoding="utf-8").read()
anc = "    # guarda: lo que trl va a entrenar debe ser exactamente la mascara construida aqui"
parche = ("    # con modelos 4-bit, trl 0.21 re-aplica prepare_model_for_kbit_training, que\n"
          "    # congela TODOS los parametros (LoRA incluida). El optimizador se crea en\n"
          "    # train(), asi que basta con reactivar los pesos LoRA aqui.\n"
          "    for n, p in trainer.model.named_parameters():\n"
          "        if \"lora_\" in n:\n"
          "            p.requires_grad_(True)\n")
if "p.requires_grad_(True)" not in s:
    assert s.count(anc) == 1
    s = s.replace(anc, parche + anc)
    open(p, "w", encoding="utf-8").write(s)
print("parche aplicado:", "p.requires_grad_(True)" in open(p, encoding="utf-8").read())

parche aplicado: True


In [12]:
# prueba corta con el modelo real: chequeo de gradiente + tiempo por paso
!cd /content/proyecto && python entrenamiento/qlora_entrenar.py --modelo Qwen/Qwen3-4B-Instruct-2507 --max-steps 3 --eval-limite 8 --salida /tmp/prueba 2>&1 | grep -v -E "Downloading|Reconstruct|Fetching|Loading weights|^\s*[0-9]+%.*it/s\]$"

datos: train=3200 dev=200 hard_dev=150 | campos=['titulo', 'hora_inicio', 'duracion_minutos', 'participantes', 'ubicacion', 'recordatorio_minutos']
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
modo de entrenamiento: 4-bit NF4 + LoRA (GPU)
tokens por ejemplo: prompt<=524 | respuesta 42-70 (lo unico con loss) | total 519-591 | max_length=599
Truncating eval dataset: 100%|██████████| 200/200 [00:00<00:00, 71617.93 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
mascara de loss verificada: 59 de 561 tokens del ejemplo 0 entrenan (solo el JSON)
gradiente verificado: 252 tensores LoRA reciben gradiente (loss inicial 1.058)
 96%|█████████▌| 24/25 [01:16<00:03,  3.24s/it]
                                           